In [1]:
import requests
import mysql.connector
import pandas as pd
from bs4 import BeautifulSoup
from sqlConfig import loginConfig
import tableSetUpMethods as tsum
import time
from concurrent.futures import ThreadPoolExecutor

In [2]:
cnx = mysql.connector.connect(
    user=loginConfig['user'],
    password=loginConfig['password'],
    host=loginConfig['host'],
    database=loginConfig['database']
)
cursor = cnx.cursor()

In [3]:
# Execute the query
query = "SELECT * FROM eventHyperlinks"
cursor.execute(query)
eventHyperLinks = cursor.fetchall()

In [40]:
def fightStatGrabber(event):
    time.sleep(1)
    eventID = event[0]
    eventName = event[1]
    eventDate = event[2]
    eventLocation = event[3]
    eventURL = event[4]
    response = requests.get(eventURL)
    soup = BeautifulSoup(response.content, 'html.parser')
    
    #grab main fights table
    table = soup.find_all('table')[0] 
    rows = table.find_all('tr')
    
    #grab all onClick links in the fighter table
    fightLinks = []
    winners = []
    for row in rows:
        onclick = row.get('onclick')
        if onclick:
            fightLinks.append(onclick)
    
        #want to find the winner from this table and store for later
        winner_tag = row.find('a', class_='b-link b-link_style_black')
        header_tag = row.find('th')
        if winner_tag and not header_tag:  # Check if the tag exists before calling get_text()
            winner = winner_tag.get_text().strip()
            winners.append(winner)

    #pair
    fightLinks = [fight.split("'")[1] for fight in fightLinks]
    winnerAndLink = list(zip(winners, fightLinks))
    eventStats = [(winner, link, eventID, eventName, eventDate, eventLocation) for winner, link in winnerAndLink]

    fightStats = []
    for event in eventStats:
        fight = event[1]
        time.sleep(1)
        response = requests.get(fight)
        soup = BeautifulSoup(response.content, 'html.parser')
        table = soup.find_all('table')[0] 
        stats = table.find_all('p', class_='b-fight-details__table-text')
        fightStats = fightStats + [p.get_text(strip=True) for p in stats]
        fightStats.extend(list(event))
    return(fightStats)

In [41]:
demoEvents = eventHyperLinks[1:2]
tmp = [fightStatGrabber(event) for event in demoEvents]

In [42]:
tmp

[['Brandon Moreno',
  'Brandon Royval',
  '0',
  '0',
  '112 of 211',
  '145 of 510',
  '53%',
  '28%',
  '119 of 219',
  '177 of 556',
  '3 of 5',
  '1 of 2',
  '60%',
  '50%',
  '0',
  '0',
  '0',
  '0',
  '2:54',
  '0:35',
  'Brandon Royval',
  'http://ufcstats.com/fight-details/74f5cbecac011abb',
  2,
  'UFC Fight Night: Moreno vs. Royval 2',
  'February 24, 2024',
  'Mexico City, Distrito Federal, Mexico',
  'Yair Rodriguez',
  'Brian Ortega',
  '1',
  '0',
  '55 of 115',
  '28 of 73',
  '47%',
  '38%',
  '64 of 135',
  '47 of 104',
  '0 of 1',
  '3 of 6',
  '0%',
  '50%',
  '0',
  '1',
  '0',
  '0',
  '1:32',
  '5:21',
  'Brian Ortega',
  'http://ufcstats.com/fight-details/3b05a4ced6dd74f5',
  2,
  'UFC Fight Night: Moreno vs. Royval 2',
  'February 24, 2024',
  'Mexico City, Distrito Federal, Mexico',
  'Daniel Zellhuber',
  'Francisco Prado',
  '0',
  '0',
  '120 of 272',
  '90 of 246',
  '44%',
  '36%',
  '120 of 272',
  '91 of 249',
  '0 of 0',
  '0 of 0',
  '---',
  '---',
 

In [7]:
#request page
eventURL = 'http://ufcstats.com/event-details/902ab9197b83d0db'
response = requests.get(eventURL)
soup = BeautifulSoup(response.content, 'html.parser')

#grab main fights table
table = soup.find_all('table')[0] 
rows = table.find_all('tr')

#grab all onClick links in the fighter table
fightLinks = []
winners = []
for row in rows:
    onclick = row.get('onclick')
    if onclick:
        fightLinks.append(onclick)

    #want to find the winner from this table and store for later
    winner_tag = row.find('a', class_='b-link b-link_style_black')
    header_tag = row.find('th')
    if winner_tag and not header_tag:  # Check if the tag exists before calling get_text()
        winner = winner_tag.get_text().strip()
        winners.append(winner)
fightLinks = [fight.split("'")[1] for fight in fightLinks]

Brandon Royval 1
Brian Ortega 2
Daniel Zellhuber 3
Yazmin Jauregui 4
Manuel Torres 5
Raoni Barcelos 6
Jesus Aguilar 7
Edgar Chairez 8
Fares Ziam 9
Ronaldo Rodriguez 10
Felipe dos Santos 11
Muhammad Naimov 12


In [14]:
winnerAndLink = list(zip(winners, fightLinks)
eventStats = [(winner, link, ) for winner, link in winnerAndLink]

[('Brandon Royval',
  'http://ufcstats.com/fight-details/74f5cbecac011abb',
  'e',
  'e'),
 ('Brian Ortega',
  'http://ufcstats.com/fight-details/3b05a4ced6dd74f5',
  'v',
  'v'),
 ('Daniel Zellhuber',
  'http://ufcstats.com/fight-details/ecbece0cbd60029e',
  'e',
  'e'),
 ('Yazmin Jauregui',
  'http://ufcstats.com/fight-details/6e4acb2aaa2cee56',
  'n',
  'n'),
 ('Manuel Torres',
  'http://ufcstats.com/fight-details/673a156925dfb31d',
  't',
  't'),
 ('Raoni Barcelos',
  'http://ufcstats.com/fight-details/063e7ce759c52f81',
  'n',
  'I'),
 ('Jesus Aguilar',
  'http://ufcstats.com/fight-details/d1d3b6ab8a56e973',
  'a',
  'D')]

In [12]:
for pair in winnerAndLink:
    winner = pair[0]
    fight = pair[1]
    time.sleep(1)
    response = requests.get(fight)
    soup = BeautifulSoup(response.content, 'html.parser')
    table = soup.find_all('table')[0] 
    stats = table.find_all('p', class_='b-fight-details__table-text')
    text_values = [p.get_text(strip=True) for p in stats]
    text_values.append(winner)
    print(text_values)

['Brandon Moreno', 'Brandon Royval', '0', '0', '112 of 211', '145 of 510', '53%', '28%', '119 of 219', '177 of 556', '3 of 5', '1 of 2', '60%', '50%', '0', '0', '0', '0', '2:54', '0:35', 'Brandon Royval']
['Yair Rodriguez', 'Brian Ortega', '1', '0', '55 of 115', '28 of 73', '47%', '38%', '64 of 135', '47 of 104', '0 of 1', '3 of 6', '0%', '50%', '0', '1', '0', '0', '1:32', '5:21', 'Brian Ortega']
['Daniel Zellhuber', 'Francisco Prado', '0', '0', '120 of 272', '90 of 246', '44%', '36%', '120 of 272', '91 of 249', '0 of 0', '0 of 0', '---', '---', '0', '0', '0', '0', '0:05', '0:25', 'Daniel Zellhuber']
['Yazmin Jauregui', 'Sam Hughes', '0', '0', '94 of 269', '83 of 197', '34%', '42%', '97 of 272', '88 of 206', '0 of 0', '0 of 6', '---', '0%', '0', '0', '0', '0', '0:00', '0:17', 'Yazmin Jauregui']
['Manuel Torres', 'Chris Duncan', '0', '0', '3 of 5', '1 of 2', '60%', '50%', '5 of 7', '1 of 2', '2 of 2', '1 of 3', '100%', '33%', '1', '0', '0', '0', '0:42', '0:40', 'Manuel Torres']
['Cristi

In [ ]:
text_values